# Negative-pH Calibration with Temperature Correction

Glass electrodes deviate strongly from Nernstian behaviour in the **acid-error /
negative-pH** region. This notebook models the response with a **logistic** function
and makes each logistic parameter **temperature-dependent** via a polynomial whose
degree is chosen by **AIC**.

Logistic response at a single temperature:

$$\text{EMF}(\text{pH}) = \dfrac{A}{1 + \exp\!\big(k\,(\text{pH}-x_0)\big)}$$

Temperature dependence:

$$A(T),\;k(T),\;x_0(T) \;=\; \text{polynomials in } T$$

whose degrees are selected automatically by **AICc** (small-sample AIC).

**Two modes:**

- **Mode A - Build calibration**: give EMF-pH curves at **>= 3 temperatures**,
  fit a logistic per temperature, then fit $A(T),k(T),x_0(T)$ with AIC-selected
  polynomial degrees, and build a full $\text{EMF}(\text{pH},T)$ model.
- **Mode B - Predict sample pH**: give a measured sample EMF **and** temperature,
  and invert the model to obtain pH (including negative pH).

*From Saleesongsom et al., ACS Omega (2026).*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.optimize import curve_fit

np.set_printoptions(suppress=True)


## Mode A - Build the temperature-dependent calibration

### Step 1: enter your calibration data

Provide one EMF-pH curve **per temperature**, for at least **3 temperatures**.
Each temperature entry is `T_in_C: (list_of_pH, list_of_EMF_mV)`.
Curves should span the acid-error region (roughly pH 1 down to negative pH).


In [ ]:
# ============ EDIT YOUR CALIBRATION DATA HERE ============
# Format:  temperature_C : ( [pH values], [EMF values in mV] )
# You need at least 3 temperatures. Each curve needs enough points
# (>= 5, ideally spanning pH ~1 down to negative pH) to fit the logistic.

calibration_data = {
    25.0: (
        [1.00, 0.50, 0.00, -0.50, -1.00, -1.50, -2.00, -2.50, -3.00, -4.00, -5.00],
        [372.0, 388.0, 405.0, 424.0, 443.0, 461.0, 476.0, 489.0, 499.0, 512.0, 519.0],
    ),
    40.0: (
        [0.97, 0.50, 0.27, -0.09, -0.39, -0.69, -1.27, -1.94, -2.67, -3.55, -4.80],
        [373.4, 399.6, 411.5, 432.0, 444.4, 453.2, 469.4, 484.3, 497.6, 511.8, 526.1],
    ),
    50.0: (
        [1.00, 0.50, 0.00, -0.50, -1.00, -1.50, -2.00, -2.50, -3.00, -4.00, -5.00],
        [388.0, 405.0, 424.0, 445.0, 465.0, 483.0, 499.0, 512.0, 523.0, 537.0, 545.0],
    ),
}
# =========================================================

temps = np.array(sorted(calibration_data.keys()), float)
assert len(temps) >= 3, "Provide data for at least 3 temperatures."
print(f"{len(temps)} temperatures loaded: {temps} C")


### Step 2: fit a logistic curve at each temperature

Extracts $A$, $k$, $x_0$ for every temperature.


In [ ]:
def logistic(pH, A, k, x0):
    return A / (1.0 + np.exp(k * (pH - x0)))

params = {"A": [], "k": [], "x0": []}
fits = {}   # T -> (A, k, x0)

for T in temps:
    pH, emf = calibration_data[T]
    pH  = np.asarray(pH, float)
    emf = np.asarray(emf, float)
    # sensible initial guesses: A slightly above the max EMF plateau
    p0 = [emf.max() * 1.03, 0.5, np.median(pH)]
    popt, _ = curve_fit(logistic, pH, emf, p0=p0, maxfev=20000)
    A, k, x0 = popt
    fits[T] = (A, k, x0)
    params["A"].append(A); params["k"].append(k); params["x0"].append(x0)
    # goodness of fit
    resid = emf - logistic(pH, *popt)
    ss = 1 - np.sum(resid**2) / np.sum((emf - emf.mean())**2)
    print(f"T = {T:5.1f} C   A = {A:7.2f}   k = {k:6.4f}   x0 = {x0:6.3f}   R^2 = {ss:.4f}")

for key in params:
    params[key] = np.asarray(params[key], float)


### Step 3: choose polynomial degree for each parameter by AIC

For every parameter series $A(T)$, $k(T)$, $x_0(T)$ we fit polynomials of increasing
degree and score them with **AICc** (corrected for small sample size). The lowest AICc
wins. You can **override** any choice by setting `DEGREE_A`, `DEGREE_K`, `DEGREE_X0`
to an integer instead of `None`.


In [ ]:
def fit_poly_aic(T, y, max_degree=3):
    '''Fit polynomials deg 1..max, return (best_degree, best_coeffs, table).
    table rows: (degree, RSS, AIC, AICc). Uses AICc for selection.'''
    T = np.asarray(T, float); y = np.asarray(y, float)
    n = len(T)
    # cap degree so AICc is defined: need n - (deg+1) - 1 > 0  ->  deg < n-2
    dmax = max(1, min(max_degree, n - 2))
    table = []
    for d in range(1, dmax + 1):
        coeffs = np.polyfit(T, y, d)
        yhat = np.polyval(coeffs, T)
        rss = float(np.sum((y - yhat)**2))
        K = (d + 1) + 1          # poly coeffs + variance
        if rss <= 0:
            aic = -np.inf
        else:
            aic = n * np.log(rss / n) + 2 * K
        denom = n - K - 1
        aicc = aic + (2 * K * (K + 1) / denom) if denom > 0 else np.inf
        table.append((d, rss, aic, aicc, coeffs))
    best = min(table, key=lambda row: row[3])   # lowest AICc
    return best[0], best[4], table


def report_aic(name, T, y, override):
    best_d, best_coeffs, table = fit_poly_aic(T, y)
    print(f"\n{name}(T):")
    print("  deg    RSS         AIC        AICc")
    for d, rss, aic, aicc, _ in table:
        mark = "  <- AIC best" if d == best_d else ""
        print(f"   {d}   {rss:10.4g}  {aic:9.3f}  {aicc:9.3f}{mark}")
    if override is not None:
        best_d = int(override)
        best_coeffs = np.polyfit(T, y, best_d)
        print(f"  -> OVERRIDDEN to degree {best_d}")
    else:
        print(f"  -> selected degree {best_d}")
    return best_d, best_coeffs


# ====== OPTIONAL DEGREE OVERRIDES (set to None to let AIC decide) ======
DEGREE_A  = None
DEGREE_K  = None
DEGREE_X0 = None
# ======================================================================

deg_A,  coef_A  = report_aic("A",  temps, params["A"],  DEGREE_A)
deg_k,  coef_k  = report_aic("k",  temps, params["k"],  DEGREE_K)
deg_x0, coef_x0 = report_aic("x0", temps, params["x0"], DEGREE_X0)

# temperature-dependent parameter functions
A_of_T  = lambda T: np.polyval(coef_A,  T)
k_of_T  = lambda T: np.polyval(coef_k,  T)
x0_of_T = lambda T: np.polyval(coef_x0, T)

print("\nFinal polynomial degrees:  A:", deg_A, " k:", deg_k, " x0:", deg_x0)


In [ ]:
# ---- Plot each parameter vs T with its selected polynomial ----
Tgrid = np.linspace(temps.min(), temps.max(), 200)
fig, axes = plt.subplots(1, 3, figsize=(10, 3), dpi=120)
for ax, name, yvals, fn, d in zip(
        axes, ["A", "k", "x0"],
        [params["A"], params["k"], params["x0"]],
        [A_of_T, k_of_T, x0_of_T],
        [deg_A, deg_k, deg_x0]):
    ax.scatter(temps, yvals, c="#E8261C", zorder=3, s=45, edgecolors="black")
    ax.plot(Tgrid, fn(Tgrid), color="#333333", lw=1.4)
    ax.set_xlabel("Temperature (°C)")
    ax.set_ylabel(f"{name}(T)")
    ax.set_title(f"{name}(T)  ·  degree {d}")
plt.tight_layout()
plt.show()


In [ ]:
# ---- Full model: EMF as a function of pH and T ----
def EMF_model(pH, T):
    return A_of_T(T) / (1.0 + np.exp(k_of_T(T) * (pH - x0_of_T(T))))

# temperature-graded family of modeled curves
pH_grid = np.linspace(-5, 1, 300)
cmap = plt.cm.autumn_r
norm = mpl.colors.Normalize(vmin=temps.min(), vmax=temps.max())

fig, ax = plt.subplots(figsize=(4.2, 3.2), dpi=120)
for T in np.linspace(temps.min(), temps.max(), 60):
    ax.plot(pH_grid, EMF_model(pH_grid, T), color=cmap(norm(T)), lw=0.9)
# overlay the raw calibration points
for T in temps:
    pH, emf = calibration_data[T]
    ax.scatter(pH, emf, s=18, color=cmap(norm(T)), edgecolors="black", linewidths=0.4, zorder=3)

ax.set_xlabel(r"pH")
ax.set_ylabel("EMF (mV)")
ax.set_title("Modeled EMF(pH, T)")
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02); cbar.set_label("Temp (°C)")
plt.tight_layout()
plt.show()


## Mode B - Predict sample pH (with temperature)

Invert the logistic model at the sample temperature:

$$\text{pH} = x_0(T) + \dfrac{1}{k(T)}\,\ln\!\left(\dfrac{A(T)}{\text{EMF}} - 1\right)$$

Enter a measured sample EMF **and** its temperature. Both single values and lists work.


In [ ]:
# ============ EDIT YOUR SAMPLE MEASUREMENTS HERE ============
sample_EMF = [450.0, 500.0]     # measured EMF of sample(s), mV
sample_T   = [40.0,  40.0]      # temperature of each sample, deg C
# ===========================================================

sample_EMF = np.atleast_1d(np.asarray(sample_EMF, float))
sample_T   = np.atleast_1d(np.asarray(sample_T, float))
if sample_T.size == 1:
    sample_T = np.full_like(sample_EMF, sample_T[0])

def predict_pH(emf, T):
    A, k, x0 = A_of_T(T), k_of_T(T), x0_of_T(T)
    ratio = A / emf - 1.0
    if ratio <= 0:
        return np.nan   # EMF at/above the A plateau - outside model range
    return x0 + np.log(ratio) / k

print("Sample EMF (mV) @ T (C)  ->  predicted pH")
for e, T in zip(sample_EMF, sample_T):
    ph = predict_pH(e, T)
    flag = "" if np.isfinite(ph) else "  (EMF outside model range)"
    print(f"  {e:8.2f} @ {T:5.1f}  ->  pH {ph:7.3f}{flag}")
